# NDVI processing for the Mujib Basin Digital Twin

This notebook processes the Sentinel-2 NDVI data described in Section 3.3.4 of the thesis.
It covers two stages: extraction from Google Earth Engine (GEE) and local raster preparation
for the dashboard.

**Stage 1 (GEE extraction):**
- Filter Sentinel-2 Level 2A surface reflectance to the Mujib Basin
- Mask clouds and cirrus using the QA60 band (bits 10 and 11)
- Compute NDVI from the 10 m NIR (B8) and Red (B4) bands
- Generate two median composites: baseline (2015-2017) and recent (2023-2025)
- Export as GeoTIFFs at 50 m resolution in UTM Zone 36N (EPSG:32636)

**Stage 2 (local raster preparation):**
- Clean nodata values and remove out-of-range observations
- Compute the NDVI change layer (recent minus baseline)
- Reproject to WGS84 on a 1655 x 2000 pixel grid
- Produce grayscale value rasters and colour-rendered rasters
- Generate metadata (bounds, value ranges, colour mapping) for the dashboard

**Inputs:** Sentinel-2 Level 2A via Google Earth Engine; Mujib Basin boundary (GEE asset)

**Outputs:** NDVI PNGs and metadata in `runtime-data/ndvi/`

**Thesis reference:** Section 3.3.4 (methodology), Section 4.4.3 (results), Equation 3.4 (NDVI formula)

**Note:** Stage 1 requires a Google Earth Engine account and authentication. The exported
GeoTIFFs are intermediate files not included in the repository. Stage 2 can be run on the
exported GeoTIFFs to reproduce the dashboard-ready outputs.


## Stage 1: Google Earth Engine extraction

The cells below extract Sentinel-2 NDVI composites from GEE. They require
`earthengine-api` and `geemap` to be installed and authenticated.
If you do not have GEE access, skip to Stage 2.


### 1.1 GEE authentication and basin boundary


In [ ]:
import ee, geemap, os
ee.Initialize(project='ee-procheta')  # Replace with your own GEE project ID

# -----------------------------
#  BASIN
# -----------------------------
mujib = ee.FeatureCollection("projects/ee-procheta/assets/Mujib_4326")  # Replace with your own GEE asset path
region = mujib.geometry()

# -----------------------------
#  TIME PERIODS
# -----------------------------
S2_BASE_START = '2015-06-01'
S2_BASE_END   = '2017-12-31'

S2_RECENT_START = '2023-01-01'
S2_RECENT_END   = '2025-06-01'

SEASON_START = '2020-01-01'
SEASON_END   = '2024-12-31'

# -----------------------------
#  SENTINEL-2 COLLECTION + CLOUD MASK
# -----------------------------
def mask_s2_sr_clouds(img):
    qa = img.select('QA60')
    cloud_bit = 1 << 10
    cirrus_bit = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit).eq(0).And(
           qa.bitwiseAnd(cirrus_bit).eq(0))
    return img.updateMask(mask).copyProperties(img, img.propertyNames())

def add_ndvi(img):
    ndvi = img.normalizedDifference(['B8','B4']).rename('NDVI')
    return img.addBands(ndvi)

s2_sr = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
         .filterBounds(mujib)
         .filterDate('2015-01-01', '2025-12-31')
         .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 40))
         .map(mask_s2_sr_clouds)
         .map(add_ndvi))

# -----------------------------
#  BASELINE, RECENT, DELTA
# -----------------------------
baseline_s2 = (s2_sr
               .filterDate(S2_BASE_START, S2_BASE_END)
               .select('NDVI')
               .mean()
               .clip(mujib))

recent_s2 = (s2_sr
             .filterDate(S2_RECENT_START, S2_RECENT_END)
             .select('NDVI')
             .mean()
             .clip(mujib))

delta_s2 = recent_s2.subtract(baseline_s2).rename('Delta_NDVI').clip(mujib)

# -----------------------------
#  SEASONAL NDVI (SPRING & SUMMER)
# -----------------------------
def filter_season(ic, month_from, month_to):
    return ic.filter(ee.Filter.calendarRange(month_from, month_to, 'month'))

season_ic = s2_sr.filterDate(SEASON_START, SEASON_END).select('NDVI')

spring_ndvi = filter_season(season_ic, 3, 5).mean().clip(mujib)   # Mar - May
summer_ndvi = filter_season(season_ic, 6, 8).mean().clip(mujib)   # Jun - Aug

# -----------------------------
#  EXPORT FUNCTION (50 m, UTM 36N)
# -----------------------------
out_dir = str(NDVI_DIR)  # Update to your local GeoTIFF export directory

os.makedirs(out_dir, exist_ok=True)

def export_geotiff_local(image, filename, region, scale=50, crs="EPSG:32636"):
    geemap.ee_export_image(
        image,
        filename=os.path.join(out_dir, filename),
        scale=scale,
        region=region,
        crs=crs
    )
    print("Saved:", os.path.join(out_dir, filename))

# -----------------------------
#  RUN EXPORTS
# -----------------------------
export_geotiff_local(baseline_s2, "S2_Baseline_NDVI_2015_2017.tif", region)
export_geotiff_local(recent_s2,   "S2_Recent_NDVI_2023_2025.tif", region)
export_geotiff_local(delta_s2,    "S2_Delta_NDVI_Recent_minus_Baseline.tif", region)

export_geotiff_local(spring_ndvi, "S2_Spring_NDVI_2020_2024.tif", region)
export_geotiff_local(summer_ndvi, "S2_Summer_NDVI_2020_2024.tif", region)


## Stage 2: Local raster preparation

This stage takes the exported GeoTIFFs and prepares them for the dashboard.
It runs locally without GEE access.


### 2.1 Configuration


In [ ]:
from pathlib import Path
import numpy as np
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
import json
from PIL import Image

# Repository paths
REPO_ROOT = Path("..")
NDVI_DIR = REPO_ROOT / "ndvi-intermediate"  # GeoTIFF exports from GEE (not in repository)
NDVI_OUT = REPO_ROOT / "runtime-data" / "ndvi"  # Dashboard-ready outputs
NDVI_ANALYSIS = REPO_ROOT / "ndvi-analysis"  # Analysis outputs

NDVI_OUT.mkdir(parents=True, exist_ok=True)


### 2.2 Clean nodata values and compute NDVI change

The baseline GeoTIFF may have nodata pixels coded as very large negative numbers.
These are replaced with NaN. The change layer is computed as recent minus baseline.


In [ ]:
import numpy as np, rasterio
from rasterio import Affine

inp = str(NDVI_DIR / "placeholder")  # Update path
out = str(NDVI_DIR / "placeholder")  # Update path

with rasterio.open(inp) as src:
    a = src.read(1).astype("float32")
    profile = src.profile.copy()

nodata = -9999.0
a_fixed = a.copy()
a_fixed[a_fixed == 0.0] = nodata

profile.update(nodata=nodata, dtype="float32", compress="deflate")

with rasterio.open(out, "w", **profile) as dst:
    dst.write(a_fixed, 1)

print("Wrote:", out)


# --- Compute NDVI change (recent minus baseline) ---
import numpy as np
import rasterio

baseline_fixed = str(NDVI_DIR / "placeholder")  # Update path
recent = str(NDVI_DIR / "placeholder")  # Update path
out = str(NDVI_DIR / "placeholder")  # Update path

nodata = -9999.0

with rasterio.open(baseline_fixed) as sb, rasterio.open(recent) as sr:
    # Read with masks
    B = sb.read(1, masked=True)
    R = sr.read(1, masked=True)

    # Optional sanity mask to keep NDVI in plausible range
    B = np.ma.masked_where((B < -1) | (B > 1), B)
    R = np.ma.masked_where((R < -1) | (R > 1), R)

    # Delta only where both are valid
    common = (~B.mask) & (~R.mask)
    D = np.ma.array(R - B, mask=~common)

    # Use recent raster as reference for georeferencing/profile
    profile = sr.profile.copy()
    profile.update(dtype="float32", nodata=nodata, compress="deflate", count=1)

    out_arr = D.filled(nodata).astype("float32")

with rasterio.open(out, "w", **profile) as dst:
    dst.write(out_arr, 1)

print("Wrote:", out)


### 2.3 Reproject to WGS84 and render dashboard PNGs

The cleaned rasters are reprojected from UTM Zone 36N to geographic coordinates (WGS84)
on a 1655 x 2000 pixel grid. Grayscale value rasters and colour-rendered rasters are
produced for the dashboard overlay system. A metadata file stores the geographic bounds,
value ranges, and colour mapping parameters.


In [ ]:
from pathlib import Path
import json
import numpy as np
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.transform import Affine
from PIL import Image

baseline_fp = NDVI_DIR / "placeholder"  # Update path
recent_fp   = NDVI_DIR / "placeholder"  # Update path
delta_fp    = NDVI_DIR / "placeholder"  # Update path

out_dir = baseline_fp.parent / "NDVI_CESIUM_PREVIEW_4326"
out_dir.mkdir(exist_ok=True)

dst_crs = "EPSG:4326"
MAX_DIM = 2000  # keep previews light for Cesium single-tile overlay

def load_arr(fp):
    with rasterio.open(fp) as src:
        arr = src.read(1).astype("float32")
        nod = src.nodata
        mask = (arr != nod) if nod is not None else np.isfinite(arr)
        return arr, mask, src

def make_ds_transform(src, dst_transform, dst_w, dst_h):
    max_dim = max(dst_w, dst_h)
    scale = max_dim / MAX_DIM if max_dim > MAX_DIM else 1.0
    out_w = int(round(dst_w / scale))
    out_h = int(round(dst_h / scale))
    dst_transform_ds = Affine(dst_transform.a * (dst_w / out_w), dst_transform.b, dst_transform.c,
                              dst_transform.d, dst_transform.e * (dst_h / out_h), dst_transform.f)
    return dst_transform_ds, out_w, out_h

b, bmask, bsrc = load_arr(baseline_fp)
r, rmask, _    = load_arr(recent_fp)
d, dmask, _    = load_arr(delta_fp)

# Use baseline mask as the basin mask
valid_mask = bmask & dmask

# Base transform/shape in 4326
dst_transform, dst_w, dst_h = calculate_default_transform(
    bsrc.crs, dst_crs, bsrc.width, bsrc.height, *bsrc.bounds
)
dst_transform_ds, out_w, out_h = make_ds_transform(bsrc, dst_transform, dst_w, dst_h)

def reproj(arr, src_transform, src_crs, dst_shape):
    dst = np.full(dst_shape, np.nan, dtype="float32")
    reproject(
        arr, dst,
        src_transform=src_transform, src_crs=src_crs,
        dst_transform=dst_transform_ds, dst_crs=dst_crs,
        resampling=Resampling.bilinear
    )
    return dst

def reproj_mask(mask, src_transform, src_crs, dst_shape):
    dst = np.zeros(dst_shape, dtype="uint8")
    reproject(
        mask.astype("uint8"), dst,
        src_transform=src_transform, src_crs=src_crs,
        dst_transform=dst_transform_ds, dst_crs=dst_crs,
        resampling=Resampling.nearest
    )
    return dst.astype(bool)

shape = (out_h, out_w)
b4326 = reproj(b, bsrc.transform, bsrc.crs, shape)
r4326 = reproj(r, bsrc.transform, bsrc.crs, shape)
d4326 = reproj(d, bsrc.transform, bsrc.crs, shape)
m4326 = reproj_mask(valid_mask, bsrc.transform, bsrc.crs, shape)

# Scales for mapping gray(0..255) -> NDVI
SCALES = {
    "baseline": {"vmin": 0.04, "vmax": 0.22},
    "recent":   {"vmin": 0.05, "vmax": 0.27},
    "delta":    {"vmin": -0.05, "vmax": 0.08},
}

def to_png(arr, mask, vmin, vmax, out_png):
    norm = (arr - vmin) / (vmax - vmin)
    gray = np.clip(np.round(norm * 255.0), 0, 255).astype(np.uint8)
    rgba = np.zeros((gray.shape[0], gray.shape[1], 4), dtype=np.uint8)
    rgba[..., :3] = gray[..., None]
    rgba[..., 3] = np.where(mask & np.isfinite(arr), 255, 0).astype(np.uint8)
    Image.fromarray(rgba).save(out_png)

to_png(b4326, m4326, **SCALES["baseline"], out_png=out_dir/"s2_baseline.png")
to_png(r4326, m4326, **SCALES["recent"],   out_png=out_dir/"s2_recent.png")
to_png(d4326, m4326, **SCALES["delta"],    out_png=out_dir/"s2_delta.png")

# Bounds in degrees for Cesium Rectangle
west = dst_transform_ds.c
north = dst_transform_ds.f
east = west + dst_transform_ds.a * out_w
south = north + dst_transform_ds.e * out_h  # e is negative

meta = {
    "west": float(west), "south": float(south), "east": float(east), "north": float(north),
    "width": out_w, "height": out_h,
    "scales": SCALES
}
(out_dir/"s2_bounds_scales.json").write_text(json.dumps(meta, indent=2))
print("Saved Cesium previews + bounds:", out_dir)


### 2.4 Generate colour-rendered PNGs

Apply colour ramps to the grayscale value PNGs for display in the dashboard.
The baseline and recent composites use a green vegetation ramp.
The change layer uses a diverging red-to-green ramp centred on zero.


In [ ]:
import json
import numpy as np
from pathlib import Path
from PIL import Image

# ---- paths (edit if needed) ----
folder = NDVI_OUT
bounds_json = folder / "s2_bounds_scales.json"

baseline_png = folder / "s2_baseline.png"
recent_png   = folder / "s2_recent.png"
delta_png    = folder / "s2_delta.png"

# output
baseline_out = folder / "s2_baseline_color.png"
recent_out   = folder / "s2_recent_color.png"
delta_out    = folder / "s2_delta_color.png"


def lerp(a, b, t):
    return a + (b - a) * t

def apply_palette(t, stops):
    """
    t: normalized 0..1 array
    stops: list of (pos, (r,g,b)) with pos in [0,1]
    returns uint8 RGB array (H,W,3)
    """
    t = np.clip(t, 0, 1)
    stops = sorted(stops, key=lambda x: x[0])

    out = np.zeros((*t.shape, 3), dtype=np.float32)

    for i in range(len(stops) - 1):
        p0, c0 = stops[i]
        p1, c1 = stops[i + 1]
        mask = (t >= p0) & (t <= p1)
        if not np.any(mask):
            continue
        tt = (t[mask] - p0) / (p1 - p0 + 1e-12)

        c0 = np.array(c0, dtype=np.float32)
        c1 = np.array(c1, dtype=np.float32)
        out[mask] = lerp(c0, c1, tt[:, None])

    # fill below first stop and above last stop
    out[t < stops[0][0]] = np.array(stops[0][1], dtype=np.float32)
    out[t > stops[-1][0]] = np.array(stops[-1][1], dtype=np.float32)

    return np.clip(out, 0, 255).astype(np.uint8)

def recolor_png(in_png, out_png, vmin, vmax, palette_stops):
    img = Image.open(in_png).convert("RGBA")
    arr = np.array(img)
    rgb = arr[..., :3]
    a   = arr[..., 3]

    # grayscale value 0..255 (use R channel)
    gray = rgb[..., 0].astype(np.float32)

    # map grayscale -> NDVI using vmin/vmax
    ndvi = vmin + (gray / 255.0) * (vmax - vmin)

    # normalize to 0..1 for palette
    t = (ndvi - vmin) / (vmax - vmin + 1e-12)

    colored = apply_palette(t, palette_stops)

    out = np.zeros_like(arr)
    out[..., :3] = colored
    out[..., 3]  = a  # keep the same transparency mask

    Image.fromarray(out, mode="RGBA").save(out_png)
    print("Saved:", out_png)

# Load vmin/vmax from your bounds json
meta = json.loads(bounds_json.read_text(encoding="utf-8"))
scales = meta["scales"]

# NDVI palette: brown -> yellow -> light green -> green
NDVI_STOPS = [
    (0.00, (120,  72,  40)),  # brown
    (0.40, (220, 200, 120)),  # tan/yellow
    (0.65, (160, 220, 140)),  # light green
    (1.00, (  0, 140,  60)),  # green
]

# Delta palette: blue -> white -> red (centered around 0)
# We'll map 0 to ~middle by choosing stops based on vmin/vmax (done below)
def delta_stops(vmin, vmax):
    zero_pos = (0 - vmin) / (vmax - vmin + 1e-12)
    zero_pos = float(np.clip(zero_pos, 0, 1))
    return [
        (0.00, ( 40,  90, 200)),          # blue
        (zero_pos, (245, 245, 245)),      # white at 0
        (1.00, (200,  40,  40)),          # red
    ]

# IMPORTANT: for fair visual comparison, use the SAME vmin/vmax for baseline & recent
# (Pick recent scale and reuse it for baseline too)
vmin_ndvi = scales["recent"]["vmin"]
vmax_ndvi = scales["recent"]["vmax"]

recolor_png(baseline_png, baseline_out, vmin_ndvi, vmax_ndvi, NDVI_STOPS)
recolor_png(recent_png,   recent_out,   vmin_ndvi, vmax_ndvi, NDVI_STOPS)

vmin_d = scales["delta"]["vmin"]
vmax_d = scales["delta"]["vmax"]
recolor_png(delta_png, delta_out, vmin_d, vmax_d, delta_stops(vmin_d, vmax_d))


### 2.5 Change detection statistics

Classify the basin into increase, stable, and decline using a threshold of +/-0.01 NDVI units.
These statistics are reported in Section 4.4.3.


In [ ]:
# Change detection with 0.01 threshold (matching thesis Section 4.4.3)
# D should be the delta NDVI array loaded from the change raster

thr = 0.01  # change threshold

# Load the delta raster
delta_path = NDVI_DIR / "Aligned_S2_Delta_FIXED_Recent_minus_Baseline.tif"
with rasterio.open(delta_path) as src:
    D = src.read(1)
    D = np.where(np.isfinite(D), D, np.nan)

valid = np.isfinite(D)
n_valid = valid.sum()

n_increase = np.sum(D[valid] > thr)
n_stable = np.sum((D[valid] >= -thr) & (D[valid] <= thr))
n_decrease = np.sum(D[valid] < -thr)

pixel_area_km2 = 50 * 50 / 1e6  # 50 m resolution

print(f"Increase (>{thr}): {n_increase:,} pixels ({100*n_increase/n_valid:.2f}%), area = {n_increase * pixel_area_km2:,.2f} km2")
print(f"Stable ({-thr} to {thr}): {n_stable:,} pixels ({100*n_stable/n_valid:.2f}%), area = {n_stable * pixel_area_km2:,.2f} km2")
print(f"Decrease (<{-thr}): {n_decrease:,} pixels ({100*n_decrease/n_valid:.2f}%), area = {n_decrease * pixel_area_km2:,.2f} km2")


## Summary

This notebook produced the NDVI layers used by the dashboard:

- `s2_baseline.png` and `s2_baseline_color.png`: baseline NDVI (2015-2017)
- `s2_recent.png` and `s2_recent_color.png`: recent NDVI (2023-2025)
- `s2_delta.png` and `s2_delta_color.png`: NDVI change (recent minus baseline)
- `s2_bounds_scales.json`: geographic bounds, value ranges, and colour mapping

Basin-level NDVI statistics (Section 4.4.3):
- Mean NDVI rose from 0.0864 (baseline) to 0.0949 (recent), an increase of 9.8%
- 30.2% of the basin showed an increase, 64.9% remained stable, 4.9% declined
- Greening was most concentrated along the northwestern highlands and western basin margin

Thesis references: Section 3.3.4 (methodology), Section 4.4.3 (results),
Equation 3.4 (NDVI formula), Figure 4.5 (condition maps), Figure 4.6 (change map).
